# TSFM: compose, run, evaluate

A walkthrough of the **compose-and-run** half of the TSFM server — the tools that actually
execute. The story: *forecast Chiller 6, and prove the result is trustworthy.*

| | tool | |
|---|---|---|
| 1 | `recipe_template` | learn what a recipe looks like |
| 2 | `profile_series` · `data_quality` | evidence before deciding |
| 3 | `find_models` · `resolve_model` | pick a model, preflight it |
| 4 | `run_recipe` | fit + backtest |
| 5 | `run_recipe` + `conformal` | prediction intervals |
| 6 | `run_recipe` + `task=anomaly` | same tool, anomaly detection |
| 7 | `run_tabular_recipe` | panel data + FLOps features |
| 8 | `run_plan` | a DAG of steps |
| 9 | `evaluate` | score across configs |
| 10 | `list_runs` · `get_run` | the ledger |

**The idea:** the server executes and records; **the agent decides**. Nothing here picks a model
or a horizon for you — the *recipe* carries every choice.

Runs on the in-memory store, so **no CouchDB needed**. Set `TSFM_STORE=couch` for the real thing.

In [ ]:
import json, os, sys

REPO = os.path.abspath(os.environ.get('AOB_REPO', '.'))
SRC  = os.path.join(REPO, 'src')
sys.path.insert(0, SRC)

# the tsfm server is spawned as a SUBPROCESS and inherits this env - sys.path only
# affects this notebook, not the child
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['TSFM_STORE'] = 'memory'        # 'couch' + COUCHDB_URL for the real store

from mcphub import ToolUniverse

SERVER_CMD = os.environ.get('SERVER_CMD', f'{sys.executable} -m servers.tsfm.main').split()
tu = ToolUniverse(servers={'tsfm': SERVER_CMD})
print(f'{tu.load_tools(servers=["tsfm"])} tools discovered')

def run(name, args=None):
    """Call a tool and unwrap FastMCP's envelope.

    A tool annotated `-> Union[X, ErrorResult]` arrives as {'result': {...}}; a bare-model
    annotation (recipe_template) arrives flat. Agents must handle both.
    """
    r = tu.run({'name': f'tsfm.{name}', 'arguments': args or {}})
    return r['result'] if isinstance(r, dict) and set(r) == {'result'} else r

def show(x, n=900):
    print(json.dumps(x, indent=2, default=str)[:n])

## Some data to work with

A synthetic chiller signal: daily seasonality + drift, with a few injected spikes.

In [ ]:
import numpy as np, pandas as pd
from servers.tsfm.io import refs

N = 300
t = np.arange(N)
signal = 10 + np.sin(t / 24 * 2 * np.pi) * 3 + 0.01 * t + np.random.RandomState(0).normal(0, .2, N)
signal[[70, 150, 240]] += 8                       # spikes, for the anomaly step

CHILLER = refs.materialize_iot(signal, asset_id='chiller6')
print('data file pointer:', CHILLER)
print(pd.read_csv(refs._path(CHILLER)).head(3).to_string(index=False))

## 1. `recipe_template` — what *is* a recipe?

The agent's first stop. A recipe is an opaque object in the schema, so this tool is how its
shape is discoverable at all. Static — it reads nothing from the catalog.

Note `estimator_spec`: an estimator is **either** a catalog `model_id` **or** an inline
`sktime_class`. That's the seam between the model catalog and the run tools.

In [ ]:
t = run('recipe_template')
print('REQUIRED:  an `estimator` or an `ensemble`\n')
print('estimator can be:');  [print('  -', x) for x in t['estimator_spec']]
print('\noptional blocks:'); [print('  -', x) for x in t['optional_blocks']]
print('\nworked examples:', list(t['examples']))

## 2. Evidence before deciding

`profile_series` returns *facts*, not recommendations — no `recommended_model` key anywhere.
`data_quality` cleans and hands back a new pointer to feed the next tool.

In [ ]:
p = run('profile_series', {'dataset_path': CHILLER, 'timestamp_column': 'timestamp'})
print('observations :', p['n_observations'])
print('channels     :', p['n_channels'])
print('dominant period:', p['dominant_period'], ' <- ~24, the daily cycle')
print('non-stationary :', p['non_stationary'], ' <- the drift')
print('\nevidence only? no recommend*/chosen* keys:',
      not any(k.startswith(('recommend', 'chosen')) for k in p))

q = run('data_quality', {'dataset_path': CHILLER, 'timestamp_column': 'timestamp'})
print(f"\ndata_quality: {q['rows_in']} -> {q['rows_out']} rows, cleaned pointer emitted")

## 3. Pick a model, then preflight it

We register two cards: the real **TTM-R1** (805K params — the smallest TS foundation model)
and a classical baseline.

`resolve_model` is the interesting one: it confirms TTM **can** be loaded and says where the
weights come from — **without downloading them**.

In [ ]:
TTM   = 'sktime.forecasting.ttm.TinyTimeMixerForecaster'
NAIVE = 'sktime.forecasting.naive.NaiveForecaster'

run('register_model', {'model': {
    'model_id': 'ttm_r1', 'task_ids': ['tsfm_forecasting'],
    'description': 'IBM Granite TinyTimeMixer R1: 805K-param tiny TS foundation forecaster.',
    'sktime_class': TTM, 'params': {'model_path': 'ibm-granite/granite-timeseries-ttm-r1'},
    'hf_repo': 'ibm-granite/granite-timeseries-ttm-r1', 'model_family': 'TinyTimeMixer',
    'context_length': 512, 'prediction_length': 96, 'domain': 'energy'}})

run('register_model', {'model': {
    'model_id': 'drift_baseline', 'task_ids': ['tsfm_forecasting'],
    'description': 'Naive drift baseline - always worth beating.',
    'sktime_class': NAIVE, 'params': {'strategy': 'drift'}, 'domain': 'energy'}})

found = run('find_models', {'task_id': 'tsfm_forecasting', 'domain': 'energy'})
print('candidates:', [m['model_id'] for m in found['models']])

r = run('resolve_model', {'model_id': 'ttm_r1'})
print('\nresolve_model(ttm_r1):')
print('  resolvable   :', r['resolvable'])
print('  weights_from :', r['weights_from'])
print('  regime       :', r['training_regime'])
print('  reason       :', r['reason'])
print('\n^ nothing was downloaded - this is a preflight')

## 4. `run_recipe` — fit and backtest

The recipe names the model and the horizon. The server does exactly what it says.

We use the classical baseline so this notebook runs anywhere; swapping in
`{'model_id': 'ttm_r1'}` would use the foundation model (and need `transformers`).

In [ ]:
r = run('run_recipe', {
    'dataset_path': CHILLER, 'timestamp_column': 'timestamp', 'target_columns': ['value'],
    'asset_id': 'chiller6',
    'recipe': {'estimator': {'model_id': 'drift_baseline'}, 'fh': [1, 2, 3, 4, 5]},
})
print('status        :', r['status'])
print('run_id        :', r['run_id'])
print('metric        :', r['metric'])
print('backtest_score:', r['backtest_score'])
print('regime        :', r['training_regime'])
print('results_file  :', r['results_file'])
BASELINE_RUN = r['run_id']

### Bulk data never crosses the MCP boundary

The tool returned a **pointer**. The forecast itself is in the file — that is how steps chain
without shipping arrays through the protocol.

In [ ]:
payload = json.load(open(refs._path(r['results_file'])))
print('keys in the results file:', sorted(payload))
print('\nforecast head:', payload['forecast_head'])

## 5. Add uncertainty: the `conformal` block

One extra key turns a point forecast into calibrated prediction intervals.

It is **expensive** — conformal refits the model repeatedly (~10s at 120 points), so this
cell uses a short slice.

*(This path used to crash: sktime returns MultiIndex columns, so the intervals came back
keyed by tuples and `json.dump` refused them. Fixed in this PR.)*

In [ ]:
# conformal refits the model repeatedly, so it is expensive: ~10s at 120 points and
# far worse at 300. Use a short slice to keep the notebook snappy.
SHORT = refs.materialize_iot(signal[:120], asset_id='chiller6_short')

r = run('run_recipe', {
    'dataset_path': SHORT, 'timestamp_column': 'timestamp', 'target_columns': ['value'],
    'recipe': {'estimator': {'model_id': 'drift_baseline'},
               'fh': [1, 2, 3],
               'conformal': {'coverage': 0.9}},
})
print('status:', r['status'])
pi = json.load(open(refs._path(r['results_file'])))['prediction_interval']
print('\n90% prediction interval:')
show(pi, 400)
print('\nkeys are strings:', all(isinstance(k, str) for k in pi),
      '<- they were tuples before the fix in this PR; json.dump refused them')

## 6. Same tool, different task: anomaly detection

`run_recipe` dispatches on `recipe['task']`. Forecasting and anomaly are **one tool**, not two.
It should find the three spikes we injected.

In [ ]:
r = run('run_recipe', {
    'dataset_path': CHILLER, 'timestamp_column': 'timestamp', 'target_columns': ['value'],
    'recipe': {'task': 'tsfm_anomaly_detection',
               'estimator': {'sktime_class': 'sktime.detection.lof.SubLOF',
                             'params': {'window_size': 24, 'n_neighbors': 5, 'novelty': True}}},
})
print('status      :', r['status'])
print('regime      :', r['training_regime'], '<- SubLOF is classical: fits on the series itself')
labels = json.load(open(refs._path(r['results_file'])))['anomaly_label']
flagged = [i for i, v in enumerate(labels) if v]
print(f'\nflagged {len(flagged)} points; we injected spikes at 70, 150, 240')
print('near a spike:', [i for i in flagged if min(abs(i-s) for s in (70,150,240)) <= 2])

## 7. `run_tabular_recipe` — panel data

Different shape entirely: each **row** is a sample, not a timestamp. It extracts FLOps features
first, then fits a tabular estimator. Omit `label_column` for clustering.

In [ ]:
rng = np.random.RandomState(0)
n, T = 40, 24
X = np.zeros((n, T)); y = np.array([0, 1] * (n // 2))
for i in range(n):
    X[i] = np.sin(np.arange(T) * (0.2 if y[i] == 0 else 0.8)) + 0.1 * rng.randn(T)
df = pd.DataFrame(X, columns=[f'f{i}' for i in range(T)]); df['label'] = y
refs._ensure_workdir()
panel = os.path.join(refs.WORKDIR, 'showcase_panel.csv'); df.to_csv(panel, index=False)

r = run('run_tabular_recipe', {
    'dataset_path': panel, 'label_column': 'label',
    'recipe': {'task': 'tsfm_classification',
               'estimator': {'sktime_class': 'sklearn.ensemble.RandomForestClassifier',
                             'params': {'n_estimators': 50}}},
})
print('task      :', r['task'])
print('metric    :', r['metric'], '=', r['cv_score'])
print('n_features:', r['n_features'], '<- FLOps extractors applied to every row')

## 8. `run_plan` — chain steps into a DAG

Steps are `{id, task, args, recipe, dep?}`, topologically sorted. A step can reference another's
output with **`@step_id`**, which resolves to that step's file pointer.

Note it is `task`, not `tool`, and `recipe` is a **sibling** of `args`.

In [ ]:
r = run('run_plan', {'asset_id': 'chiller6', 'plan_spec': {'steps': [
    {'id': 'baseline', 'task': 'forecast', 'args': {'data_ref': CHILLER},
     'recipe': {'estimator': {'sktime_class': NAIVE, 'params': {'strategy': 'last'}},
                'fh': [1, 2, 3]}},
    {'id': 'drift', 'task': 'forecast', 'dep': ['baseline'], 'args': {'data_ref': CHILLER},
     'recipe': {'estimator': {'sktime_class': NAIVE, 'params': {'strategy': 'drift'}},
                'fh': [1, 2, 3]}},
]}})
print('status :', r['status'])
print('plan_id:', r['plan_id'])
for sid, out in r['outputs'].items():
    print(f"  {sid:9s} -> {out['ref']}")
    print(f"            {out['summary']}")

## 9. `evaluate` — score across configs

GIFT-Eval **style**: MASE + CRPS normalised against a seasonal-naive baseline, geometric mean
across configs. A local computation — not the public benchmark.

Configs carry the series inline here (it scores many at once), unlike the file-pointer tools.

In [ ]:
r = run('evaluate', {
    'recipe': {'estimator': {'sktime_class': NAIVE, 'params': {'strategy': 'drift'}}},
    'configs': [
        {'name': 'daily',  'y': list(signal[:120]), 'fh': [1, 2, 3], 'sp': 24},
        {'name': 'weekly', 'y': list(signal[:200]), 'fh': [1, 2, 3], 'sp': 24},
    ],
})
print('status:', r['status'])
print('\nper config:'); show(r['per_config'], 500)
print('\naggregate:');  show(r['agg'], 300)

## 10. The ledger — everything above was recorded

Every run wrote a record. Without these the run tools would be write-only.

In [ ]:
runs = run('list_runs', {'asset_id': 'chiller6'})
print(f"{len(runs['runs'])} runs, {len(runs['plans'])} plans recorded for chiller6\n")
for x in runs['runs']:
    print(f"  {x['run_id']:18s} {x.get('task','?'):24s} score={x.get('backtest_score')}")

one = run('get_run', {'run_id': BASELINE_RUN})
print(f"\nget_run({BASELINE_RUN}):")
print('  metric:', one.get('metric'), '| score:', one.get('backtest_score'))
print('  recipe it ran:', json.dumps(one.get('recipe'), default=str)[:90])

## Recap

```
recipe_template  ->  learn the contract
profile_series   ->  evidence, not decisions
find_models      ->  choose from the catalog
resolve_model    ->  preflight, without downloading
run_recipe       ->  fit + backtest      (+conformal -> intervals)
                     task=anomaly -> same tool, dense labels
run_tabular_recipe -> panel data + FLOps features
run_plan         ->  a DAG, chained by @step_id pointers
evaluate         ->  score across configs
list_runs/get_run -> the ledger
```

Two properties worth keeping in mind:

* **The agent decides, the server executes.** No tool here chooses a model or a horizon.
* **Bulk data never crosses MCP.** Pointers in, pointers out — that is what makes a plan
  chainable without shipping arrays through the protocol.

In [ ]:
tu.close()
print('closed')